# 📥 Step 2a: Lakeflow Ingestion — Auto Loader (Files → Bronze)

Raw security telemetry lands as **JSON files** in the Unity Catalog volume
`rajarora_febar.soc_alert_triage.landing` (one folder per day, like a SIEM/EDR export).
**Auto Loader** (`cloudFiles`) incrementally ingests only *new* files into the bronze table
`raw_security_events`, tracking progress in a checkpoint so every file is loaded exactly once.

- Explicit schema + `_rescued_data` column: malformed/unexpected fields are captured, not dropped
- Lineage columns: `source_file`, `ingested_at`
- `availableNow` trigger: process everything new, then stop (runs as a task in the Lakeflow Job)

In [0]:
%run ./01_config

# ⚙️ Configuration — SOC Alert Triage

Shared configuration for all notebooks. Update the catalog name below before running.

✅ Configuration loaded
   Catalog: rajarora_febar
   Schema:  soc_alert_triage
   Tables:  10 tables configured


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, BooleanType

raw_event_schema = StructType([
    StructField("event_id", StringType()),
    StructField("event_timestamp", StringType()),
    StructField("source_type", StringType()),
    StructField("customer_id", StringType()),
    StructField("asset_id", StringType()),
    StructField("severity_score", LongType()),
    StructField("is_true_positive", BooleanType()),
    StructField("raw_payload", StringType()),
])

def bronze_snapshot():
    if not spark.catalog.tableExists(TABLES["raw_security_events"]):
        return 0, 0
    r = spark.table(TABLES["raw_security_events"]).agg(
        F.count("*").alias("rows"), F.countDistinct("source_file").alias("files")).first()
    return r.rows, r.files

def run_autoloader():
    """Ingest all files not yet processed, then stop. Returns (new_rows, new_files)."""
    rows_before, files_before = bronze_snapshot()
    query = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.partitionColumns", "")      # event_day folders are for organisation only
        .option("rescuedDataColumn", "_rescued_data")
        .schema(raw_event_schema)
        .load(LANDING_PATH)
        .select("*",
                F.col("_metadata.file_path").alias("source_file"),
                F.current_timestamp().alias("ingested_at"))
        .writeStream
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(TABLES["raw_security_events"])
    )
    query.awaitTermination()
    rows_after, files_after = bronze_snapshot()
    return rows_after - rows_before, files_after - files_before

In [0]:
new_rows, new_files = run_autoloader()
print(f"✅ Auto Loader run 1: ingested {new_rows:,} events from {new_files} new files")
print(f"   Source: {LANDING_PATH}")
print(f"   Target: {TABLES['raw_security_events']}")

✅ Auto Loader run 1: ingested 500,000 events from 30 new files
   Source: /Volumes/rajarora_febar/soc_alert_triage/landing/security_events
   Target: rajarora_febar.soc_alert_triage.raw_security_events


In [0]:
new_rows, new_files = run_autoloader()
print(f"✅ Auto Loader run 2: ingested {new_rows:,} events from {new_files} new files "
      "(expected 0 — already-processed files are skipped via the checkpoint)")

✅ Auto Loader run 2: ingested 0 events from 0 new files (expected 0 — already-processed files are skipped via the checkpoint)


In [0]:
spark.sql(f"""
    SELECT COUNT(*)                                                  AS total_events,
           COUNT(DISTINCT source_file)                               AS source_files,
           COUNT(DISTINCT event_id)                                  AS distinct_event_ids,
           SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_rescued_data,
           SUM(CASE WHEN event_id IS NULL OR event_timestamp IS NULL THEN 1 ELSE 0 END) AS rows_missing_keys,
           MIN(event_timestamp)                                      AS first_event,
           MAX(event_timestamp)                                      AS last_event
    FROM {TABLES['raw_security_events']}
""").display()

spark.sql(f"""
    SELECT regexp_extract(source_file, 'event_day=([0-9-]+)', 1) AS event_day,
           COUNT(*) AS events, MIN(ingested_at) AS ingested_at
    FROM {TABLES['raw_security_events']}
    GROUP BY 1 ORDER BY 1
    LIMIT 10
""").display()

print("Next step: Run 02b_silver_gold_transform")

total_events,source_files,distinct_event_ids,rows_with_rescued_data,rows_missing_keys,first_event,last_event
500000,30,500000,0,0,2026-09-01T00:00:00Z,2026-09-30T23:59:56Z


event_day,events,ingested_at
2026-09-01,16991,2026-10-12T21:48:05.705Z
2026-09-02,16860,2026-10-12T21:48:05.705Z
2026-09-03,16730,2026-10-12T21:48:05.705Z
2026-09-04,16704,2026-10-12T21:48:05.705Z
2026-09-05,16462,2026-10-12T21:48:05.705Z
2026-09-06,16547,2026-10-12T21:48:05.705Z
2026-09-07,16634,2026-10-12T21:48:05.705Z
2026-09-08,16699,2026-10-12T21:48:05.705Z
2026-09-09,16787,2026-10-12T21:48:05.705Z
2026-09-10,16661,2026-10-12T21:48:05.705Z


Next step: Run 02b_silver_gold_transform
